# Project: Automated Exploratory Data Analysis with Pandas

Build an automated EDA pipeline that profiles datasets, detects missing values, finds correlations, and generates summary reports.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print('Pandas version:', pd.__version__)

## Generate Synthetic Dataset

In [ ]:
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'id': range(n),
    'age': np.random.normal(35, 12, n).astype(int),
    'income': np.random.lognormal(10.5, 0.8, n).astype(int),
    'education': np.random.choice(['High School','Bachelor','Master','PhD'], n, p=[0.3,0.4,0.2,0.1]),
    'city': np.random.choice(['NYC','LA','Chicago','Houston','Phoenix'], n),
    'score': np.random.uniform(0, 100, n),
    'signup_date': pd.date_range('2020-01-01', periods=n, freq='D'),
    'is_active': np.random.choice([True, False], n, p=[0.7, 0.3]),
})
df.loc[np.random.random(n) < 0.05, 'income'] = np.nan
df.loc[np.random.random(n) < 0.03, 'age'] = np.nan
df.loc[np.random.random(n) < 0.02, 'city'] = np.nan
print('Dataset shape:', df.shape)
df.head()

## 1. Data Profile

In [ ]:
def data_profile(df):
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'n_missing': df.isnull().sum(),
        'pct_missing': (df.isnull().sum() / len(df) * 100).round(2),
        'n_unique': df.nunique(),
    })
    profile['memory_mb'] = df.memory_usage(deep=True) / 1024**2
    return profile

profile = data_profile(df)
profile

## 2. Descriptive Statistics by Type

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols].describe().T

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().head(10).to_string())

## 3. Correlation Analysis

In [ ]:
corr = df[num_cols].corr()
print('Correlation matrix:')
print(corr.round(3))

high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i,j])
             for i in range(len(corr.columns))
             for j in range(i+1, len(corr.columns)) if abs(corr.iloc[i,j]) > 0.3]
if high_corr:
    print('\nHighly correlated pairs:')
    for p in high_corr:
        print(f'  {p[0]} vs {p[1]}: {p[2]:.3f}')

## 4. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    missing.plot(kind='bar', figsize=(10, 4))
    plt.ylabel('Count')
plt.title('Missing Values by Column')
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
    row_missing = df.isnull().sum(axis=1)
    print(f'Rows with 1+ missing: {(row_missing > 0).sum()} ({((row_missing>0).sum()/len(df)*100):.1f}%)')
else:
    print('No missing values found.')

## 5. Outlier Detection (IQR Method)

In [ ]:
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return len(outliers), lower, upper

for col in num_cols:
    if col == 'id': continue
    n_out, lower, upper = detect_outliers_iqr(df, col)
    print(f'{col}: {n_out} outliers (bounds: [{lower:.1f}, {upper:.1f}])')

## 6. Automated EDA Report

In [ ]:
def generate_eda_report(df):
    print('='*60)
print('EDA REPORT'.center(60))
print('='*60)
    print(f'\nShape: {df.shape[0]} rows x {df.shape[1]} cols')
    print(f'Memory: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
    print(f'Duplicates: {df.duplicated().sum()}')
    print(f'Missing: {df.isnull().sum().sum()} ({df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:.2f}%)')

generate_eda_report(df)

## Summary

- Profiles data types, missing values, and memory
- Computes descriptive statistics
- Analyzes correlations
- Detects outliers using the IQR method
- Generates a comprehensive EDA report